# 🎵 MoodBeats AI — Session 7: Sessions & Supabase

Today we give MoodBeats **memory**. We'll track who uses the app and save their mood history.

---
## Why HTTP Has No Memory

HTTP is **stateless** — each request is completely independent. The server doesn't know if it's
talking to the same person it talked to 5 seconds ago.

```
Request 1: GET /  →  Server responds: "Who are you? I don't know."
Request 2: GET /detect  →  Server: "Who are you? I still don't know."
```

**Sessions** fix this: the server gives the browser a signed cookie (a unique ID), and the
browser sends it with every request. The server reads the cookie to identify the user.

---

In [ ]:
!pip install fastapi uvicorn itsdangerous starlette supabase nest-asyncio -q
import nest_asyncio; nest_asyncio.apply()
print("Ready!")

---
## Part 1: Session Cookies with Starlette

`SessionMiddleware` adds a signed cookie to every response and reads it from every request.
`request.session` is a dict — read and write it like any Python dict.

In [ ]:
import uuid
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from starlette.middleware.sessions import SessionMiddleware

app = FastAPI()
app.add_middleware(
    SessionMiddleware,
    secret_key="my-super-secret-key-change-in-production",
    max_age=86_400 * 30   # 30 days
)

def get_or_create_session_id(request: Request) -> str:
    """Return the session's UUID, creating one if it doesn't exist."""
    if "session_id" not in request.session:
        request.session["session_id"] = str(uuid.uuid4())
    return request.session["session_id"]

@app.get("/whoami")
def whoami(request: Request):
    session_id = get_or_create_session_id(request)
    visits     = request.session.get("visits", 0) + 1
    request.session["visits"] = visits
    return {"session_id": session_id, "visits": visits}

import threading, uvicorn
threading.Thread(target=lambda: uvicorn.run(app, port=8010, log_level="error"), daemon=True).start()
print("Server running on port 8010")

In [ ]:
import requests as req

# Use a session to preserve cookies across requests
s = req.Session()

r1 = s.get("http://localhost:8010/whoami")
print("Visit 1:", r1.json())

r2 = s.get("http://localhost:8010/whoami")
print("Visit 2:", r2.json())  # visits should be 2, same session_id

# New session (no cookie) → new session_id, visits=1
r3 = req.get("http://localhost:8010/whoami")
print("New session:", r3.json())

### ✏️ In-Class Exercise 7a — Save Display Name to Session

In [ ]:
from fastapi import Form
from fastapi.responses import RedirectResponse

# TODO: Add a POST /profile/update route that:
#   1. Reads 'display_name' from form data
#   2. Saves it to request.session['display_name']
#   3. Redirects to /whoami

@app.post("/profile/update")
async def profile_update(request: Request, display_name: str = Form(...)):
    # TODO: save display_name to session and redirect
    pass

# TODO: Update /whoami to also return request.session.get('display_name', 'Anonymous')

---
## Part 2: Supabase — Cloud Database Setup

### Steps to set up Supabase:
1. Go to **https://supabase.com** and sign up (free)
2. Click **"New project"** — give it a name (e.g. `moodbeats-ai`)
3. Go to **SQL Editor** and run the SQL below to create the table
4. Go to **Settings → API** to get your `Project URL` and `anon` key
5. Add them to Colab Secrets as `SUPABASE_URL` and `SUPABASE_KEY`

In [ ]:
# Copy and paste this SQL into Supabase → SQL Editor → New Query → Run

create_table_sql = """
CREATE TABLE IF NOT EXISTS mood_history (
    id          BIGSERIAL PRIMARY KEY,
    session_id  TEXT        NOT NULL,
    input_type  TEXT        DEFAULT 'text',
    emotion     TEXT        NOT NULL,
    confidence  FLOAT,
    description TEXT,
    songs       JSONB,
    detected_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_mood_history_session ON mood_history (session_id);
"""

print("Copy the SQL above into Supabase SQL Editor and click Run")
print(create_table_sql)

---
## Part 3: Saving and Retrieving Mood History

In [ ]:
from google.colab import userdata
from supabase import create_client, Client

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

# Only initialise if credentials are available
db: Client | None = None
if SUPABASE_URL and SUPABASE_KEY:
    db = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Supabase connected!")
else:
    print("Supabase not configured — skipping DB calls")

In [ ]:
def save_mood(session_id: str, input_type: str, result: dict) -> bool:
    """Save a mood detection to Supabase. Returns True on success."""
    if not db:
        return False  # graceful: app works without DB
    try:
        db.table("mood_history").insert({
            "session_id":  session_id,
            "input_type":  input_type,
            "emotion":     result.get("emotion", "neutral"),
            "confidence":  result.get("confidence", 0.0),
            "description": result.get("mood_description", ""),
            "songs":       result.get("songs", []),
        }).execute()
        return True
    except Exception as e:
        print(f"save_mood error: {e}")
        return False

def get_history(session_id: str) -> list:
    """Get all detections for a session, newest first."""
    if not db:
        return []
    try:
        resp = (
            db.table("mood_history")
              .select("*")
              .eq("session_id", session_id)
              .order("detected_at", desc=True)
              .execute()
        )
        return resp.data
    except Exception as e:
        print(f"get_history error: {e}")
        return []

# Test saving a mock result
mock_session = str(uuid.uuid4())
mock_result = {
    "emotion": "happy",
    "confidence": 0.91,
    "mood_description": "Feeling great!",
    "songs": [{"title": "Happy", "artist": "Pharrell", "genre": "Pop"}]
}

success = save_mood(mock_session, "text", mock_result)
print(f"Save result: {success}")

history = get_history(mock_session)
print(f"History rows: {len(history)}")
if history:
    print(f"First row emotion: {history[0]['emotion']}")

### ✏️ In-Class Exercise 7b — Build the History Page Template

In [ ]:
import os
os.makedirs("templates", exist_ok=True)

with open("templates/history.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html><head>
<title>MoodBeats — History</title>
<script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-950 text-white min-h-screen p-8">
<h1 class="text-3xl font-bold mb-6">Your Mood History</h1>

{% if not history %}
<p class="text-gray-400">No history yet. <a href="/detect" class="text-indigo-400 hover:underline">Detect your first mood</a></p>
{% else %}
<p class="text-gray-400 mb-6">{{ history | length }} detection(s) found</p>

{% for entry in history %}
<div class="bg-gray-800 rounded-xl p-5 mb-4 border border-gray-700">
    <div class="flex items-center gap-4 mb-3">
        <!-- TODO: Display the emotion name and confidence -->
        <span class="text-xl font-bold text-indigo-400">{{ entry.emotion | title }}</span>
        <span class="text-gray-400 text-sm">{{ (entry.confidence * 100) | round | int }}% confidence</span>
        <span class="text-gray-500 text-xs ml-auto">{{ entry.detected_at }}</span>
    </div>
    <p class="text-gray-300 text-sm mb-3">{{ entry.description }}</p>
    <!-- TODO: Show the first 3 songs as small pills -->
    <div class="flex flex-wrap gap-2">
    {% for song in entry.songs[:3] %}
        <span class="bg-gray-700 text-gray-300 text-xs px-3 py-1 rounded-full">
            {{ song.title }} — {{ song.artist }}
        </span>
    {% endfor %}
    </div>
</div>
{% endfor %}
{% endif %}

</body></html>
""")

print("history.html created!")

### ✏️ In-Class Exercise 7c — Render History Page

In [ ]:
from jinja2 import Environment, FileSystemLoader

# Render with mock history data
mock_history = [
    {
        "emotion": "happy",
        "confidence": 0.92,
        "description": "Feeling joyful and energetic!",
        "detected_at": "2025-01-15 14:30:00",
        "songs": [
            {"title": "Happy", "artist": "Pharrell"},
            {"title": "Uptown Funk", "artist": "Bruno Mars"},
        ]
    },
    {
        "emotion": "calm",
        "confidence": 0.85,
        "description": "Peaceful and relaxed.",
        "detected_at": "2025-01-14 10:00:00",
        "songs": [{"title": "Weightless", "artist": "Marconi Union"}]
    }
]

env = Environment(loader=FileSystemLoader("templates"))
tmpl = env.get_template("history.html")
html = tmpl.render(history=mock_history)

with open("preview_history.html", "w") as f:
    f.write(html)

print("Saved preview_history.html — download and open in browser!")

---
## 🏠 After-Class Exercises

---
### After-Class A: Profile Page Stats

In [ ]:
def compute_stats(history: list) -> dict:
    """Compute stats from history list."""
    if not history:
        return {"total": 0, "most_common": None, "face_count": 0, "text_count": 0}

    # TODO: Count total detections
    total = len(history)

    # TODO: Find the most common emotion
    # Hint: use a dict to count each emotion, then max(counts, key=counts.get)
    emotion_counts = {}
    for entry in history:
        e = entry["emotion"]
        emotion_counts[e] = emotion_counts.get(e, 0) + 1
    most_common = max(emotion_counts, key=emotion_counts.get)

    # TODO: Count face vs text detections
    face_count = sum(1 for e in history if e.get("input_type") == "face")
    text_count = total - face_count

    return {"total": total, "most_common": most_common,
            "face_count": face_count, "text_count": text_count}

# Test
stats = compute_stats(mock_history)
print(stats)

### After-Class B: Delete History

In [ ]:
def delete_history(session_id: str) -> bool:
    """Delete all history for a session from Supabase."""
    if not db:
        return False
    # TODO: Use db.table('mood_history').delete().eq('session_id', session_id).execute()
    pass

# If Supabase is configured, test it
if db:
    result = delete_history(mock_session)
    print(f"Delete result: {result}")
    print(f"History after delete: {len(get_history(mock_session))} rows")

### After-Class C (Challenge): Emotion Distribution Chart

Use the `history` data to print a simple text-based bar chart.

In [ ]:
def print_emotion_chart(history: list):
    """Print a simple ASCII bar chart of emotion frequencies."""
    counts = {}
    for entry in history:
        e = entry["emotion"]
        counts[e] = counts.get(e, 0) + 1

    max_count = max(counts.values()) if counts else 1
    print("\nEmotion Breakdown:")
    for emotion, count in sorted(counts.items(), key=lambda x: -x[1]):
        bar = "█" * int(count / max_count * 20)
        print(f"  {emotion:15} {bar} {count}")

# Test with a longer mock history
extended_history = mock_history * 3 + [{"emotion": "sad", "confidence": 0.8,
    "description": "...", "detected_at": "", "songs": []}]
print_emotion_chart(extended_history)

---
## ✅ Session 7 Complete!

**You learned:**
- Why HTTP is stateless and how sessions add memory
- `SessionMiddleware` for signed cookie sessions
- Supabase setup, table creation, and Python client
- `save_mood()` and `get_history()` with graceful fallback
- History page template with Jinja2 loops

**Next session:** Deploy it to the internet!